In [17]:
"""
MobileViT for 11-channel archaeology patches.

Dataset structure expected on Kaggle:

FINALCNNDATA_FINAL/
├── train/
│   ├── no_site/
│   └── site/
├── val/
│   ├── no_site/
│   └── site/
├── test/
│   ├── no_site/
│   └── site/
├── metadata/
│   ├── individual/
│   └── combined/
└── _split_reports/

no_site = no-site patch
site = site patch

Final 11-channel order used by this code:
1  Blue
2  Green
3  Red
4  NIR
5  LRM
6  Slope
7  SVF
8  Hillshade 1
9  Hillshade 2
10 Hillshade 3
11 Hillshade 4

The model itself only sees channel positions. The TIFF files must already use
this exact order.
"""

from pathlib import Path
import copy
import random

import numpy as np
import rasterio
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)


# =============================================================================
# 1. CONFIGURATION
# =============================================================================
class Config:
    # ------------------------- dataset -------------------------
    DATASET_PATH = Path(
        "/kaggle/input/datasets/shreyansdeshpande/cnnfinale/FINALCNNDATA_FINAL"
    )

    NUM_CHANNELS = 11
    RAW_SIZE = 250
    MODEL_SIZE = 256

    CHANNEL_ORDER = [
        "Blue",
        "Green",
        "Red",
        "NIR",
        "LRM",
        "Slope",
        "SVF",
        "Hillshade 1",
        "Hillshade 2",
        "Hillshade 3",
        "Hillshade 4",
    ]

    # ------------------------- training -------------------------
    BATCH_SIZE = 16
    NUM_EPOCHS = 30
    LEARNING_RATE = 3e-4
    WEIGHT_DECAY = 1e-4
    DROPOUT = 0.10
    NUM_WORKERS = 2
    SEED = 42

    # ------------------------- MobileViT -------------------------
    STEM_OUT_CHANNELS = 16

    STAGE1_CFG = (4, 32, 1)
    STAGE2_DOWN_CFG = (4, 64, 2)
    STAGE2_REPEAT_CFG = (4, 64, 1)
    STAGE2_REPEATS = 2

    STAGE3_DOWN_CFG = (4, 96, 2)
    STAGE4_DOWN_CFG = (4, 128, 2)
    STAGE5_DOWN_CFG = (4, 160, 2)

    MVIT3_CFG = {
        "in_channels": 96,
        "num_blocks": 2,
        "projection_dim": 144,
        "num_heads": 2,
    }
    MVIT4_CFG = {
        "in_channels": 128,
        "num_blocks": 4,
        "projection_dim": 192,
        "num_heads": 2,
    }
    MVIT5_CFG = {
        "in_channels": 160,
        "num_blocks": 3,
        "projection_dim": 240,
        "num_heads": 2,
    }

    PATCH_SIZE = 2
    MLP_RATIO = 2.0
    HEAD_CHANNELS = 640

    # ------------------------- augmentation -------------------------
    AUGMENT = True


cfg = Config()


# =============================================================================
# 2. REPRODUCIBILITY
# =============================================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(cfg.SEED)


# =============================================================================
# 3. DISPLAY CONFIGURATION
# =============================================================================
print("=" * 70)
print("MobileViT 11-channel archaeology classifier")
print("=" * 70)
print("Dataset:", cfg.DATASET_PATH)
print("Channel order:")
for i, name in enumerate(cfg.CHANNEL_ORDER, start=1):
    print(f"  Channel {i:2d} -> {name}")
print()


# =============================================================================
# 4. DATASET DISCOVERY
print("USING UPDATED DATASET LOADER: no_site/site")
# =============================================================================
def collect_records(split_name):
    records = []
    split_path = cfg.DATASET_PATH / split_name

    if not split_path.exists():
        raise FileNotFoundError(
            f"Missing split folder:\n{split_path}\n\n"
            f"Expected train, val and test folders inside DATASET_PATH."
        )

    class_mapping = {
        "no_site": 0,
        "site": 1,
    }

    for class_name, label in class_mapping.items():
        class_path = split_path / class_name

        if not class_path.exists():
            raise FileNotFoundError(
                f"Missing class folder:\n{class_path}"
            )

        tif_files = sorted(
            list(class_path.glob("*.tif"))
            + list(class_path.glob("*.tiff"))
        )

        for file_path in tif_files:
            records.append((file_path, label))

    return records


train_records = collect_records("train")
validation_records = collect_records("validation")
test_records = collect_records("test")

if not train_records or not validation_records or not test_records:
    raise ValueError(
        "At least one split contains no TIFF patches. "
        "Check the Kaggle dataset path and train/val/test/no_site/site folders."
    )

print("Training patches:  ", len(train_records))
print("Validation patches:", len(validation_records))
print("Testing patches:   ", len(test_records))

for split_name, records in (
    ("train", train_records),
    ("val", validation_records),
    ("test", test_records),
):
    num_no_site = sum(label == 0 for _, label in records)
    num_site = sum(label == 1 for _, label in records)
    print(f"{split_name:>5}: no_site = {num_no_site}, site = {num_site}")


# =============================================================================
# 5. CHANNEL STATISTICS — TRAINING SET ONLY
# =============================================================================
def calculate_channel_stats(records):
    channel_sum = np.zeros(cfg.NUM_CHANNELS, dtype=np.float64)
    channel_squared_sum = np.zeros(cfg.NUM_CHANNELS, dtype=np.float64)
    channel_pixel_count = np.zeros(cfg.NUM_CHANNELS, dtype=np.int64)

    for file_path, _ in records:
        with rasterio.open(file_path) as raster:
            image = raster.read(masked=True).astype(np.float64)

        if image.shape[0] != cfg.NUM_CHANNELS:
            raise ValueError(
                f"{file_path} contains {image.shape[0]} channels, "
                f"but the model expects {cfg.NUM_CHANNELS}."
            )

        for channel_index in range(cfg.NUM_CHANNELS):
            valid_values = image[channel_index].compressed()
            valid_values = valid_values[np.isfinite(valid_values)]

            if valid_values.size == 0:
                continue

            channel_sum[channel_index] += valid_values.sum()
            channel_squared_sum[channel_index] += np.square(valid_values).sum()
            channel_pixel_count[channel_index] += valid_values.size

    if np.any(channel_pixel_count == 0):
        missing = np.where(channel_pixel_count == 0)[0] + 1
        raise ValueError(
            f"No valid pixels found in channel(s): {missing.tolist()}"
        )

    means = channel_sum / channel_pixel_count
    variances = (
        channel_squared_sum / channel_pixel_count
        - np.square(means)
    )
    variances = np.maximum(variances, 1e-12)
    stds = np.sqrt(variances)

    return means.astype(np.float32), stds.astype(np.float32)


channel_means, channel_stds = calculate_channel_stats(train_records)

print("\nTraining-set channel statistics:")
for i in range(cfg.NUM_CHANNELS):
    print(
        f"Channel {i + 1:2d} ({cfg.CHANNEL_ORDER[i]:12s}) -> "
        f"mean={channel_means[i]:.6f}, std={channel_stds[i]:.6f}"
    )


# =============================================================================
# 6. IMAGE PREPROCESSING
# =============================================================================
def augment_image(image):
    """Apply the same spatial augmentation to every channel."""
    if np.random.rand() < 0.5:
        image = np.flip(image, axis=2).copy()  # horizontal flip

    if np.random.rand() < 0.5:
        image = np.flip(image, axis=1).copy()  # vertical flip

    rotations = np.random.randint(0, 4)
    if rotations:
        image = np.rot90(
            image,
            k=rotations,
            axes=(1, 2),
        ).copy()

    return image


def pad_to_model_size(image):
    """Pad 250x250 -> 256x256 using edge replication."""
    channels, height, width = image.shape

    if channels != cfg.NUM_CHANNELS:
        raise ValueError(
            f"Expected {cfg.NUM_CHANNELS} channels, got {channels}."
        )

    if height == cfg.RAW_SIZE and width == cfg.RAW_SIZE:
        image = np.pad(
            image,
            ((0, 0), (3, 3), (3, 3)),
            mode="edge",
        )
    elif height == cfg.MODEL_SIZE and width == cfg.MODEL_SIZE:
        pass
    else:
        raise ValueError(
            f"Expected a {cfg.RAW_SIZE}x{cfg.RAW_SIZE} or "
            f"{cfg.MODEL_SIZE}x{cfg.MODEL_SIZE} patch, got {height}x{width}."
        )

    return image


# =============================================================================
# 7. PYTORCH DATASET
# =============================================================================
class ArchaeologyDataset(Dataset):
    def __init__(
        self,
        records,
        channel_means,
        channel_stds,
        augment=False,
    ):
        self.records = records
        self.augment = augment
        self.channel_means = channel_means.reshape(
            cfg.NUM_CHANNELS, 1, 1
        ).astype(np.float32)
        self.channel_stds = channel_stds.reshape(
            cfg.NUM_CHANNELS, 1, 1
        ).astype(np.float32)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        file_path, label = self.records[index]

        with rasterio.open(file_path) as raster:
            image = raster.read(masked=True).astype(np.float32)

        if image.shape[0] != cfg.NUM_CHANNELS:
            raise ValueError(
                f"{file_path.name} has {image.shape[0]} channels; "
                f"expected {cfg.NUM_CHANNELS}."
            )

        image = image.filled(np.nan)

        # Replace invalid/NoData pixels with the training mean for that channel.
        for channel_index in range(cfg.NUM_CHANNELS):
            invalid = ~np.isfinite(image[channel_index])
            image[channel_index][invalid] = self.channel_means[
                channel_index, 0, 0
            ]

        # Augmentation is spatial, so every channel is transformed together.
        if self.augment:
            image = augment_image(image)

        # Z-score normalize each channel using training-set statistics.
        image = (
            image - self.channel_means
        ) / self.channel_stds

        # MobileViT receives 256x256.
        image = pad_to_model_size(image)

        image_tensor = torch.from_numpy(image).float()
        label_tensor = torch.tensor(label, dtype=torch.float32)

        return image_tensor, label_tensor


train_dataset = ArchaeologyDataset(
    train_records,
    channel_means,
    channel_stds,
    augment=cfg.AUGMENT,
)

validation_dataset = ArchaeologyDataset(
    validation_records,
    channel_means,
    channel_stds,
    augment=False,
)

test_dataset = ArchaeologyDataset(
    test_records,
    channel_means,
    channel_stds,
    augment=False,
)


# =============================================================================
# 8. DATALOADERS
# =============================================================================
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

pin_memory = device.type == "cuda"
from torch.utils.data import WeightedRandomSampler

train_labels_list = [label for _, label in train_records]
class_counts = np.bincount(train_labels_list)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for label in train_labels_list]

sampler = WeightedRandomSampler(
    sample_weights, num_samples=len(sample_weights), replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    sampler=sampler,          # replaces shuffle=True
    num_workers=cfg.NUM_WORKERS,
    pin_memory=pin_memory,
)


validation_loader = DataLoader(
    validation_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=pin_memory,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=pin_memory,
)

print("\nUsing device:", device)


# =============================================================================
# 9. MOBILEVIT MODEL
# =============================================================================
def conv_block(in_channels, out_channels, kernel_size=3, stride=2):
    return nn.Sequential(
        nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size,
            stride=stride,
            padding=kernel_size // 2,
            bias=False,
        ),
        nn.BatchNorm2d(out_channels),
        nn.SiLU(),
    )


class InvertedResidualBlock(nn.Module):
    """MobileNetV2-style inverted residual block."""

    def __init__(self, in_channels, expansion_ratio, out_channels, stride=1):
        super().__init__()

        expanded_channels = in_channels * expansion_ratio
        self.use_residual = (
            stride == 1 and in_channels == out_channels
        )

        self.expand = nn.Sequential(
            nn.Conv2d(
                in_channels,
                expanded_channels,
                kernel_size=1,
                bias=False,
            ),
            nn.BatchNorm2d(expanded_channels),
            nn.SiLU(),
        )

        self.depthwise = nn.Sequential(
            nn.Conv2d(
                expanded_channels,
                expanded_channels,
                kernel_size=3,
                stride=stride,
                padding=1,
                groups=expanded_channels,
                bias=False,
            ),
            nn.BatchNorm2d(expanded_channels),
            nn.SiLU(),
        )

        self.project = nn.Sequential(
            nn.Conv2d(
                expanded_channels,
                out_channels,
                kernel_size=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        m = self.expand(x)
        m = self.depthwise(m)
        m = self.project(m)

        if self.use_residual:
            m = m + x

        return m


class TransformerBlock(nn.Module):
    """Pre-norm Transformer encoder block."""

    def __init__(self, dim, num_heads, mlp_ratio, dropout):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        hidden_dim = int(dim * mlp_ratio)

        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x1 = self.norm1(x)
        attn_out, _ = self.attn(
            x1,
            x1,
            x1,
            need_weights=False,
        )
        x = x + attn_out

        x2 = self.norm2(x)
        x = x + self.mlp(x2)

        return x


class MobileViTBlock(nn.Module):
    """Local CNN -> unfold -> Transformer -> fold -> fusion."""

    def __init__(
        self,
        in_channels,
        num_blocks,
        projection_dim,
        num_heads=2,
        patch_size=2,
        mlp_ratio=2.0,
        dropout=0.1,
    ):
        super().__init__()

        self.patch_size = patch_size

        # Local representation.
        self.local_rep = nn.Sequential(
            conv_block(
                in_channels,
                in_channels,
                kernel_size=3,
                stride=1,
            ),
            nn.Conv2d(
                in_channels,
                projection_dim,
                kernel_size=1,
            ),
        )

        # L Transformer blocks.
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(
                projection_dim,
                num_heads,
                mlp_ratio,
                dropout,
            )
            for _ in range(num_blocks)
        ])

        self.norm = nn.LayerNorm(projection_dim, eps=1e-6)

        # Convert Transformer representation back to CNN feature channels.
        self.proj_back = conv_block(
            projection_dim,
            in_channels,
            kernel_size=1,
            stride=1,
        )

        # Fuse local CNN features + Transformer features.
        self.fusion = conv_block(
            2 * in_channels,
            in_channels,
            kernel_size=3,
            stride=1,
        )

    def forward(self, x):
        local_features = self.local_rep(x)
        batch_size, channels, height, width = local_features.shape
        p = self.patch_size

        if height % p != 0 or width % p != 0:
            raise ValueError(
                f"MobileViT feature map {height}x{width} is not divisible "
                f"by patch_size={p}."
            )

        # -------------------- UNFOLD --------------------
        x_unfold = local_features.reshape(
            batch_size,
            channels,
            height // p,
            p,
            width // p,
            p,
        )
        x_unfold = x_unfold.permute(
            0, 3, 5, 2, 4, 1
        )

        num_patches = (height // p) * (width // p)
        x_unfold = x_unfold.reshape(
            batch_size,
            p * p,
            num_patches,
            channels,
        )

        # Each p*p patch position is treated as part of the batch.
        x_seq = x_unfold.reshape(
            batch_size * p * p,
            num_patches,
            channels,
        )

        # -------------------- TRANSFORMER --------------------
        for block in self.transformer_blocks:
            x_seq = block(x_seq)

        x_seq = self.norm(x_seq)

        # -------------------- FOLD --------------------
        x_fold = x_seq.reshape(
            batch_size,
            p,
            p,
            height // p,
            width // p,
            channels,
        )
        x_fold = x_fold.permute(
            0, 5, 3, 1, 4, 2
        )
        x_fold = x_fold.reshape(
            batch_size,
            channels,
            height,
            width,
        )

        folded = self.proj_back(x_fold)

        # -------------------- FUSION --------------------
        out = torch.cat([x, folded], dim=1)
        out = self.fusion(out)

        return out


class MobileViT(nn.Module):
    def __init__(self):
        super().__init__()

        # Input: (B, 11, 256, 256)
        self.stem = conv_block(
            cfg.NUM_CHANNELS,
            cfg.STEM_OUT_CHANNELS,
            kernel_size=3,
            stride=2,
        )

        # 256 -> 128
        exp, out_ch, stride = cfg.STAGE1_CFG
        self.stage1 = InvertedResidualBlock(
            cfg.STEM_OUT_CHANNELS,
            exp,
            out_ch,
            stride,
        )
        prev_ch = out_ch

        # 128 -> 64, then 2 MV2 blocks at 64.
        exp, out_ch, stride = cfg.STAGE2_DOWN_CFG
        self.stage2_down = InvertedResidualBlock(
            prev_ch,
            exp,
            out_ch,
            stride,
        )
        prev_ch = out_ch

        exp, rep_ch, rep_stride = cfg.STAGE2_REPEAT_CFG
        self.stage2 = nn.Sequential(*[
            InvertedResidualBlock(
                prev_ch,
                exp,
                rep_ch,
                rep_stride,
            )
            for _ in range(cfg.STAGE2_REPEATS)
        ])
        prev_ch = rep_ch

        # 64 -> 32, MobileViT L=2.
        exp, out_ch, stride = cfg.STAGE3_DOWN_CFG
        self.stage3_down = InvertedResidualBlock(
            prev_ch,
            exp,
            out_ch,
            stride,
        )
        self.stage3_mvit = MobileViTBlock(
            patch_size=cfg.PATCH_SIZE,
            mlp_ratio=cfg.MLP_RATIO,
            dropout=cfg.DROPOUT,
            **cfg.MVIT3_CFG,
        )
        prev_ch = out_ch

        # 32 -> 16, MobileViT L=4.
        exp, out_ch, stride = cfg.STAGE4_DOWN_CFG
        self.stage4_down = InvertedResidualBlock(
            prev_ch,
            exp,
            out_ch,
            stride,
        )
        self.stage4_mvit = MobileViTBlock(
            patch_size=cfg.PATCH_SIZE,
            mlp_ratio=cfg.MLP_RATIO,
            dropout=cfg.DROPOUT,
            **cfg.MVIT4_CFG,
        )
        prev_ch = out_ch

        # 16 -> 8, MobileViT L=3.
        exp, out_ch, stride = cfg.STAGE5_DOWN_CFG
        self.stage5_down = InvertedResidualBlock(
            prev_ch,
            exp,
            out_ch,
            stride,
        )
        self.stage5_mvit = MobileViTBlock(
            patch_size=cfg.PATCH_SIZE,
            mlp_ratio=cfg.MLP_RATIO,
            dropout=cfg.DROPOUT,
            **cfg.MVIT5_CFG,
        )
        prev_ch = out_ch

        # Head: Conv 1x1 -> global average pool -> linear.
        self.conv_head = conv_block(
            prev_ch,
            cfg.HEAD_CHANNELS,
            kernel_size=1,
            stride=1,
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(cfg.HEAD_CHANNELS, 1)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)

        x = self.stage2_down(x)
        x = self.stage2(x)

        x = self.stage3_down(x)
        x = self.stage3_mvit(x)

        x = self.stage4_down(x)
        x = self.stage4_mvit(x)

        x = self.stage5_down(x)
        x = self.stage5_mvit(x)

        x = self.conv_head(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        logits = self.classifier(x)

        return logits


# =============================================================================
# 10. MODEL / LOSS / OPTIMIZER
# =============================================================================
model = MobileViT().to(device)

num_no_site = sum(label == 0 for _, label in train_records)
num_site = sum(label == 1 for _, label in train_records)

if num_site == 0:
    raise ValueError("Training set contains no class-1 patches.")

pos_weight_value = 1
pos_weight = torch.tensor(
    [pos_weight_value],
    dtype=torch.float32,
    device=device,
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = optim.AdamW(
    model.parameters(),
    lr=cfg.LEARNING_RATE,
    weight_decay=cfg.WEIGHT_DECAY,
)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=3,
)

num_parameters = sum(
    parameter.numel() for parameter in model.parameters()
)

print("\nModel parameters:", f"{num_parameters / 1e6:.2f}M")
print("Positive-class weight:", f"{pos_weight_value:.4f}")


# =============================================================================
# 11. SANITY CHECK BEFORE TRAINING
# =============================================================================
images, labels = next(iter(train_loader))
print("\nSanity check:")
print("Batch image shape:", tuple(images.shape))
print("Batch label shape:", tuple(labels.shape))

if tuple(images.shape[1:]) != (
    cfg.NUM_CHANNELS,
    cfg.MODEL_SIZE,
    cfg.MODEL_SIZE,
):
    raise ValueError(
        f"Expected batch shape (B, {cfg.NUM_CHANNELS}, "
        f"{cfg.MODEL_SIZE}, {cfg.MODEL_SIZE}), "
        f"got {tuple(images.shape)}"
    )

with torch.no_grad():
    test_output = model(images.to(device))

print("Model output shape:", tuple(test_output.shape))

if tuple(test_output.shape[1:]) != (1,):
    raise ValueError(
        f"Expected model output shape (B, 1), got {tuple(test_output.shape)}"
    )


# =============================================================================
# 12. TRAINING / VALIDATION FUNCTIONS
# =============================================================================
def run_epoch(model, loader, training):
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    context = torch.enable_grad() if training else torch.no_grad()

    with context:
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).view(-1, 1)

            if training:
                optimizer.zero_grad(set_to_none=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size

            probabilities = torch.sigmoid(outputs)
            predictions = (probabilities >= 0.5).long()
            total_correct += (
                predictions == labels.long()
            ).sum().item()
            total_samples += batch_size

    average_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    return average_loss, accuracy


# =============================================================================
# 13. TRAIN
# =============================================================================
best_val_loss = float("inf")
best_model_state = copy.deepcopy(model.state_dict())

train_losses = []
validation_losses = []
train_accuracies = []
validation_accuracies = []

print("\n" + "=" * 70)
print("TRAINING")
print("=" * 70)

for epoch in range(cfg.NUM_EPOCHS):
    train_loss, train_accuracy = run_epoch(
        model,
        train_loader,
        training=True,
    )

    validation_loss, validation_accuracy = run_epoch(
        model,
        validation_loader,
        training=False,
    )

    scheduler.step(validation_loss)

    train_losses.append(train_loss)
    validation_losses.append(validation_loss)
    train_accuracies.append(train_accuracy)
    validation_accuracies.append(validation_accuracy)

    current_lr = optimizer.param_groups[0]["lr"]

    if validation_loss < best_val_loss:
        best_val_loss = validation_loss
        best_model_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch + 1:02d}/{cfg.NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy * 100:.2f}% | "
        f"Val Loss: {validation_loss:.4f} | "
        f"Val Acc: {validation_accuracy * 100:.2f}% | "
        f"LR: {current_lr:.6g}"
    )


# Restore best validation-loss model.
model.load_state_dict(best_model_state)


# =============================================================================
# 14. COLLECT PROBABILITIES
# =============================================================================
def collect_predictions(model, loader):
    model.eval()
    from sklearn.metrics import classification_report
         print(classification_report(test_labels, (test_probabilities >= 0.5).astype(int)))
    all_labels = []
    all_probabilities = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            outputs = model(images)
            probabilities = torch.sigmoid(outputs).squeeze(1)

            all_labels.extend(
                labels.numpy().astype(int)
            )
            all_probabilities.extend(
                probabilities.cpu().numpy()
            )

    return (
        np.asarray(all_labels),
        np.asarray(all_probabilities),
    )


validation_labels, validation_probabilities = collect_predictions(
    model,
    validation_loader,
)

test_labels, test_probabilities = collect_predictions(
    model,
    test_loader,
)


# =============================================================================
# 15. CHOOSE BEST THRESHOLD ON VALIDATION SET
# =============================================================================
thresholds = np.linspace(0.01, 0.99, 99)
validation_f1_scores = []

for threshold in thresholds:
    predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    validation_f1_scores.append(
        f1_score(
            validation_labels,
            predictions,
            zero_division=0,
        )
    )

best_threshold = float(
    thresholds[np.argmax(validation_f1_scores)]
)


# =============================================================================
# 16. TEST METRICS
# =============================================================================
def calculate_metrics(labels, probabilities, threshold):
    predictions = (
        probabilities >= threshold
    ).astype(int)

    metrics = {
        "threshold": threshold,
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            labels,
            predictions,
            zero_division=0,
        ),
        "confusion_matrix": confusion_matrix(
            labels,
            predictions,
            labels=[0, 1],
        ),
    }

    if len(np.unique(labels)) == 2:
        metrics["roc_auc"] = roc_auc_score(
            labels,
            probabilities,
        )
    else:
        metrics["roc_auc"] = float("nan")

    return metrics


def print_metrics(title, metrics):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    print(f"Threshold : {metrics['threshold']:.2f}")
    print(f"Accuracy  : {metrics['accuracy']:.4f}")
    print(f"Precision : {metrics['precision']:.4f}")
    print(f"Recall    : {metrics['recall']:.4f}")
    print(f"F1 Score  : {metrics['f1']:.4f}")
    print(f"ROC-AUC   : {metrics['roc_auc']:.4f}")
    print("Confusion matrix:")
    print(metrics["confusion_matrix"])


results_050 = calculate_metrics(
    test_labels,
    test_probabilities,
    0.50,
)

results_best = calculate_metrics(
    test_labels,
    test_probabilities,
    best_threshold,
)

print_metrics(
    "TEST RESULTS — THRESHOLD 0.50",
    results_050,
)

print_metrics(
    f"TEST RESULTS — BEST VALIDATION THRESHOLD ({best_threshold:.2f})",
    results_best,
)


# =============================================================================
# 17. SAVE MODEL + NORMALIZATION STATS
# =============================================================================
model_save_path = "/kaggle/working/mobilevit_11channel_best.pth"
stats_save_path = "/kaggle/working/mobilevit_11channel_channel_stats.npz"

checkpoint = {
    "model_state_dict": model.state_dict(),
    "num_channels": cfg.NUM_CHANNELS,
    "channel_order": cfg.CHANNEL_ORDER,
    "model_size": cfg.MODEL_SIZE,
    "best_threshold": best_threshold,
}

torch.save(checkpoint, model_save_path)

np.savez(
    stats_save_path,
    means=channel_means,
    stds=channel_stds,
)

print("\nSaved model:", model_save_path)
print("Saved channel statistics:", stats_save_path)
print("Best validation threshold:", f"{best_threshold:.2f}")

MobileViT 11-channel archaeology classifier
Dataset: /kaggle/input/datasets/shreyansdeshpande/cnnfinale/FINALCNNDATA_FINAL
Channel order:
  Channel  1 -> Blue
  Channel  2 -> Green
  Channel  3 -> Red
  Channel  4 -> NIR
  Channel  5 -> LRM
  Channel  6 -> Slope
  Channel  7 -> SVF
  Channel  8 -> Hillshade 1
  Channel  9 -> Hillshade 2
  Channel 10 -> Hillshade 3
  Channel 11 -> Hillshade 4

USING UPDATED DATASET LOADER: no_site/site
Training patches:   2549
Validation patches: 523
Testing patches:    545
train: no_site = 1737, site = 812
  val: no_site = 412, site = 111
 test: no_site = 372, site = 173

Training-set channel statistics:
Channel  1 (Blue        ) -> mean=0.035586, std=0.008933
Channel  2 (Green       ) -> mean=0.043991, std=0.016917
Channel  3 (Red         ) -> mean=0.035915, std=0.014176
Channel  4 (NIR         ) -> mean=0.254984, std=0.099669
Channel  5 (LRM         ) -> mean=0.015495, std=0.724850
Channel  6 (Slope       ) -> mean=7.437615, std=6.348506
Channel  7 (

Exception in thread Thread-28 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resource_

KeyboardInterrupt: 

In [4]:
import os

statsPath = "/kaggle/working/channel_stats.npz"

if os.path.exists(statsPath):
    os.remove(statsPath)
    print("Deleted stale channel_stats.npz")

Deleted stale channel_stats.npz


In [16]:
import os

print("Contents of /kaggle/input:")
print(os.listdir("/kaggle/input/datasets/shreyansdeshpande/cnnfinale/FINALCNNDATA_FINAL"))

# Verify full path dynamically
input_base = "/kaggle/input"
for item in os.listdir(input_base):
    if "cnnfinale" in item.lower():
        full_path = os.path.join(input_base, item, "FINALCNNDATA_FINAL")
        print(f"Found dataset at: {full_path}")
        print("Subfolders:", os.listdir(full_path))

Contents of /kaggle/input:
['_split_reports', 'validation', 'test', 'train']
